<a href="https://colab.research.google.com/github/ririfka08/books-analysis-dashboard/blob/main/books_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas plotly

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('/content/books.csv')

print("Jumlah baris & kolom:", df.shape)
df.head()

Jumlah baris & kolom: (52478, 25)


,bookId,title,series,author,rating,description,language,isbn,genres,characters,bookFormat,edition,pages,publisher,publishDate,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",Hardcover,First Edition,374,Scholastic Press,09/14/08,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,English,9780439358071,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...","['Sirius Black', 'Draco Malfoy', 'Ron Weasley'...",Paperback,US Edition,870,Scholastic Inc.,09/28/04,06/21/03,['Bram Stoker Award for Works for Young Reader...,2507623,"['1593642', '637516', '222366', '39573', '14526']",98.0,['Hogwarts School of Witchcraft and Wizardry (...,https://i.gr-assets.com/images/S/compressed.ph...,2632233,26923,7.38
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,English,9999999999999,"['Classics', 'Fiction', 'Historical Fiction', ...","['Scout Finch', 'Atticus Finch', 'Jem Finch', ...",Paperback,NaN,324,Harper Perennial Modern Classics,05/23/06,07/11/60,"['Pulitzer Prize for Fiction (1961)', 'Audie A...",4501075,"['2363896', '1333153', '573280', '149952', '80...",95.0,"['Maycomb, Alabama (United States)']",https://i.gr-assets.com/images/S/compressed.ph...,2269402,23328,NaN
3,1885.Pride_and_Prejudice,Pride and Prejudice,NaN,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,English,9999999999999,"['Classics', 'Fiction', 'Romance', 'Historical...","['Mr. Bennet', 'Mrs. Bennet', 'Jane Bennet', '...",Paperback,"Modern Library Classics, USA / CAN",279,Modern Library,10/10/00,01/28/13,[],2998241,"['1617567', '816659', '373311', '113934', '767...",94.0,"['United Kingdom', 'Derbyshire, England (Unite...",https://i.gr-assets.com/images/S/compressed.ph...,1983116,20452,NaN
4,41865.Twilight,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,English,9780316015844,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...","['Edward Cullen', 'Jacob Black', 'Laurent', 'R...",Paperback,NaN,501,"Little, Brown and Company",09/06/06,10/05/05,"['Georgia Peach Book Award (2007)', 'Buxtehude...",4964519,"['1751460', '1113682', '1008686', '542017', '5...",78.0,"['Forks, Washington (United States)', 'Phoenix...",https://i.gr-assets.com/images/S/compressed.ph...,1459448,14874,2.1


In [ ]:
# cek data dlu
print(df.info())
print("\n--- Missing values per kolom ---")
print(df.isnull().sum())
print("\n--- Duplicate rows ---")
print(df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52478 entries, 0 to 52477
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bookId            52478 non-null  object 
 1   title             52478 non-null  object 
 2   series            23470 non-null  object 
 3   author            52478 non-null  object 
 4   rating            52478 non-null  float64
 5   description       51140 non-null  object 
 6   language          48672 non-null  object 
 7   isbn              52478 non-null  object 
 8   genres            52478 non-null  object 
 9   characters        52478 non-null  object 
 10  bookFormat        51005 non-null  object 
 11  edition           4955 non-null   object 
 12  pages             50131 non-null  object 
 13  publisher         48782 non-null  object 
 14  publishDate       51598 non-null  object 
 15  firstPublishDate  31152 non-null  object 
 16  awards            52478 non-null  object

In [ ]:
# Data Cleaning
import ast

# 1. Drop duplicate rows
df = df.drop_duplicates()

# 2. Drop baris yang rating-nya kosong
df = df.dropna(subset=['rating'])

In [ ]:
# 3. Parse kolom genres & awards -> ubah dari string jadi list Python asli
#    Formatnya string kayak "['Fiction', 'Classics', 'Romance']"
def parse_list_column(val):
    if pd.isna(val):
        return []
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

df['genre_list'] = df['genres'].apply(parse_list_column)
df['award_list'] = df['awards'].apply(parse_list_column)


In [ ]:
# 4. Buang buku yang genre-nya kosong setelah parsing (nggak bisa dianalisis per genre)
df = df[df['genre_list'].apply(len) > 0]

# 5. Kolom 'pages' bertipe object (ada teks seperti "352 pages"), extract angkanya
df['pages_clean'] = df['pages'].astype(str).str.extract(r'(\d+)').astype(float)

# 6. Buang buku dengan halaman kosong, 0, atau ekstrem (>5000 = kemungkinan data error)
df = df[(df['pages_clean'] > 0) & (df['pages_clean'] <= 5000)]

In [ ]:
# 7. Parse publishDate jadi datetime, ambil tahunnya buat analisis tren
df['publish_year'] = pd.to_datetime(df['publishDate'], errors='coerce').dt.year

# 8. Bersihin kolom language (samakan case & whitespace, biar "english" dan "English" nyatu)
df['language'] = df['language'].str.strip().str.title()

# 9. Bikin flag boolean: apakah buku ini pernah menang award
df['has_award'] = df['award_list'].apply(len) > 0

print("Shape setelah cleaning:", df.shape)
print("\nContoh genre_list:", df['genre_list'].iloc[0])
print("Contoh pages_clean:", df['pages_clean'].iloc[0])
print("Contoh publish_year:", df['publish_year'].iloc[0])
print("Contoh has_award:", df['has_award'].iloc[0])


/tmp/ipykernel_1613/495387604.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['publish_year'] = pd.to_datetime(df['publishDate'], errors='coerce').dt.year


Shape setelah cleaning: (46077, 30)

Contoh genre_list: ['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']
Contoh pages_clean: 374.0
Contoh publish_year: 2008.0
Contoh has_award: True


In [ ]:
# Explode Genre (1 baris = 1 genre per buku)
# Ini penting karena satu buku bisa punya banyak genre sekaligus

df_exploded = df.explode('genre_list')
df_exploded['genre_list'] = df_exploded['genre_list'].str.strip()
df_exploded.head()

,bookId,title,series,author,rating,description,language,isbn,genres,characters,bookFormat,edition,pages,publisher,publishDate,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price,genre_list,award_list,pages_clean,publish_year,has_award
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",Hardcover,First Edition,374,Scholastic Press,09/14/08,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09,Young Adult,[Locus Award Nominee for Best Young Adult Book...,374.0,2008.0,True
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",Hardcover,First Edition,374,Scholastic Press,09/14/08,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09,Fiction,[Locus Award Nominee for Best Young Adult Book...,374.0,2008.0,True
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",Hardcover,First Edition,374,Scholastic Press,09/14/08,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09,Dystopia,[Locus Award Nominee for Best Young Adult Book...,374.0,2008.0,True
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",Hardcover,First Edition,374,Scholastic Press,09/14/08,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09,Fantasy,[Locus Award Nominee for Best Young Adult Book...,374.0,2008.0,True
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",Hardcover,First Edition,374,Scholastic Press,09/14/08,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09,Science Fiction,[Locus Award Nominee for Best Young Adult Book...,374.0,2008.0,True


In [ ]:
# Analysis #1 - Genre Popularity vs Rating
# Jawab pertanyaan: genre apa yang demand tinggi DAN kualitas terjaga?

genre_stats = df_exploded.groupby('genre_list').agg(
    avg_rating=('rating', 'mean'),
    total_books=('title', 'count')
).reset_index()

# Filter genre yang punya minimal 50 buku biar nggak bias sama genre niche
genre_stats = genre_stats[genre_stats['total_books'] >= 50]
genre_stats = genre_stats.sort_values('avg_rating', ascending=False)

genre_stats.head(15)

,genre_list,avg_rating,total_books
198,Comic Strips,4.457586,58
173,Church,4.307826,92
792,Shonen,4.297606,142
65,Anime,4.280862,116
169,Christian Non Fiction,4.246250,232
563,Manga,4.244812,746
806,Social Justice,4.239673,153
781,Seinen,4.236667,84
201,Comics Manga,4.234111,343
512,Lds,4.232252,111


In [ ]:
# Visualization #1 - Scatter Plot
fig = px.scatter(
    genre_stats,
    x='total_books',
    y='avg_rating',
    text='genre_list',
    size='total_books',
    title='Genre Popularity vs Average Rating',
    labels={'total_books': 'Jumlah Buku', 'avg_rating': 'Rating Rata-rata'}
)
fig.update_traces(textposition='top center')
fig.show()



In [ ]:
#Analysis #2 - Pages vs Rating
# Jawab pertanyaan: apakah panjang buku mempengaruhi rating?

df['page_category'] = pd.cut(
    df['pages_clean'],
    bins=[0, 200, 400, 600, 10000],
    labels=['Short (<200)', 'Medium (200-400)', 'Long (400-600)', 'Very Long (600+)']
)

fig2 = px.box(df, x='page_category', y='rating',
              title='Rating Distribution by Book Length')
fig2.show()

In [ ]:
#Analysis #3 - Trend Over Time
# Jawab pertanyaan: apakah buku lama lebih "timeless" atau buku baru makin baik?
# Filter tahun yang masuk akal aja (buang outlier tahun aneh/null)

yearly_trend = df[(df['publish_year'] >= 1900) & (df['publish_year'] <= 2024)]
yearly_trend = yearly_trend.groupby('publish_year')['rating'].mean().reset_index()

fig3 = px.line(yearly_trend, x='publish_year', y='rating',
                title='Average Rating Trend by Publish Year')
fig3.show()

In [ ]:
#Analysis #4 - Awards Pattern
# Jawab pertanyaan: apakah buku yang menang award punya pola beda dari yang nggak?

award_comparison = df.groupby('has_award').agg(
    avg_rating=('rating', 'mean'),
    avg_pages=('pages_clean', 'mean'),
    total_books=('title', 'count')
).reset_index()

print(award_comparison)

   has_award  avg_rating   avg_pages  total_books
0      False    4.001974  329.390936        35635
1       True    3.984138  354.989753        10442


In [ ]:
#Analysis #5 - Underserved Language Markets
# Jawab pertanyaan: bahasa mana yang rating-nya tinggi tapi jumlah bukunya masih sedikit?

language_stats = df.groupby('language').agg(
    avg_rating=('rating', 'mean'),
    total_books=('title', 'count')
).reset_index()

# Filter bahasa yang punya minimal 20 buku (biar cukup representatif),
# lalu urutkan yang rating tinggi tapi jumlah buku sedikit
language_stats = language_stats[language_stats['total_books'] >= 20]
language_stats = language_stats.sort_values(['avg_rating', 'total_books'], ascending=[False, True])
print(language_stats.head(15))


                 language  avg_rating  total_books
33  Greek, Modern (1453-)    4.305376           93
27     Filipino; Pilipino    4.259394           33
39               Japanese    4.257176           85
13              Bulgarian    4.218070           57
45                  Malay    4.215818           55
58                Russian    4.176548           84
70                   Urdu    4.171389           36
10                Bengali    4.164247           73
16               Croatian    4.161364           22
67                Turkish    4.098122          181
24               Estonian    4.031500           20
43             Lithuanian    4.030769           39
57               Romanian    4.020513           78
17                  Czech    4.013864           44
22                English    3.999320        38816


In [ ]:
#Save Cleaned Data (buat dipakai di dashboard nanti)
df.to_csv('books_cleaned.csv', index=False)
from google.colab import files
files.download('books_cleaned.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>